# Qualidade — resultados das regras (`quality.dq_resultados`)

Lê o **log de eventos** do pipeline `olist.silver` e grava, em `olist.quality.dq_resultados`, quantos registros **passaram** e quantos **reprovaram** em cada regra, por execução (*update*) do pipeline.

Responde à pergunta de negócio: *"por tabela e por regra, quantos registros passaram, foram corrigidos e foram para a quarentena?"* (desenho, seção 2).

**Parâmetros**

| Widget | Padrão | Uso |
|---|---|---|
| `pipeline_name` | `olist.silver` | nome do pipeline; o notebook descobre o identificador sozinho |

**Regras**

- O log de eventos é a fonte oficial das métricas das expectations. Cada linha do log traz as regras de uma tabela e as contagens de uma execução.
- A carga é um `MERGE` pela chave `(update_id, dataset, regra)`: rodar o notebook duas vezes não duplica nada (ADR 0003).
- `registros_reprovados` conta **todos** os registros que violaram a regra naquela execução, inclusive duplicatas do mesmo registro lido da bronze. Por isso o número pode ser maior que o de linhas distintas na quarentena. As regras de aviso (`expect`), como `payment_gateway_preenchido`, também aparecem aqui, mas o registro continua na silver.

Desenho: `docs/design/fase-2-silver-qualidade.md` (etapa 2.4).

In [0]:
# O QUE FAZ: cria o parâmetro pipeline_name, exibido como caixa de texto no topo do notebook.
#
# FUNÇÕES:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto (se já existe, nada
#   muda). O valor é lido com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("pipeline_name", "olist.silver", "Nome do pipeline")

In [0]:
# O QUE FAZ: descobre o identificador do pipeline pelo nome, valida o formato e define os nomes da
#   tabela de destino. Falha com mensagem clara se o pipeline não existir.
#
# FUNÇÕES NATIVAS E DA BIBLIOTECA:
# - databricks.sdk.WorkspaceClient(): cliente da API do Databricks; dentro do workspace, usa a
#   identidade de quem executa o notebook, sem senha.
# - w.pipelines.list_pipelines(filter=...): lista os pipelines; "name LIKE 'x'" filtra pelo nome.
# - dbutils.widgets.get(nome): lê o valor do widget.
# - re.fullmatch(padrão, texto): verifica se o texto inteiro combina com a expressão regular.
#   Aqui garante que o id é um UUID antes de entrar numa consulta SQL (evita injeção de SQL).
# - next(iterador, padrão): primeiro item do iterador, ou o padrão se estiver vazio.
import re

from databricks.sdk import WorkspaceClient

CATALOG = "olist"
TABELA_DQ = f"{CATALOG}.quality.dq_resultados"

PIPELINE_NAME = dbutils.widgets.get("pipeline_name").strip()
pipeline = next(
    iter(WorkspaceClient().pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")), None
)
if pipeline is None:
    raise ValueError(f"Pipeline '{PIPELINE_NAME}' não encontrado.")

PIPELINE_ID = pipeline.pipeline_id
if not re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", PIPELINE_ID):
    raise ValueError("Identificador de pipeline com formato inesperado.")

print(f"pipeline: {PIPELINE_NAME} ({PIPELINE_ID})")
print(f"destino : {TABELA_DQ}")

## Tabela de destino

Uma linha por execução, tabela de dados (`dataset`) e regra. Criada uma vez (`IF NOT EXISTS`), com comentários nas colunas (convenção do projeto).

In [0]:
# O QUE FAZ: cria olist.quality.dq_resultados se ainda não existir. Rodar de novo não altera a tabela.
#
# FUNÇÕES NATIVAS:
# - spark.sql(texto): executa um comando SQL.
# - CREATE TABLE IF NOT EXISTS: cria só se a tabela não existe.
# - COMMENT '...': documentação da tabela e de cada coluna.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_DQ} (
  pipeline_id           STRING    COMMENT 'Identificador do pipeline que gerou as métricas',
  update_id             STRING    COMMENT 'Identificador da execução (update) do pipeline',
  atualizado_em         TIMESTAMP COMMENT 'Momento da última medição da regra nessa execução',
  dataset               STRING    COMMENT 'Dataset que aplicou a regra (view <entidade>_validos)',
  entidade              STRING    COMMENT 'Entidade de negócio (dataset sem o sufixo _validos)',
  regra                 STRING    COMMENT 'Nome da regra de qualidade',
  registros_aprovados   BIGINT    COMMENT 'Registros que cumpriram a regra nessa execução',
  registros_reprovados  BIGINT    COMMENT 'Registros que violaram a regra nessa execução (inclui duplicatas da bronze)',
  carregado_em          TIMESTAMP COMMENT 'Momento em que a linha foi carregada nesta tabela'
)
COMMENT 'Resultado das regras de qualidade da silver, por execução do pipeline, tabela e regra. Fonte: log de eventos do pipeline.'
""")

## Leitura do log de eventos e carga

`event_log('<id>')` é uma função de tabela que devolve os eventos do pipeline. As métricas ficam no evento `flow_progress`, num JSON com a lista de regras; o notebook abre esse JSON em uma linha por regra e soma as contagens de cada `(execução, dataset, regra)`.

In [0]:
# O QUE FAZ: lê as métricas das regras no log de eventos, consolida por execução, dataset e regra,
#   e grava com MERGE (insere o que é novo e atualiza o que já existia). Mostra quantas linhas
#   foram lidas e quantas a tabela tem no final.
#
# FUNÇÕES NATIVAS:
# - event_log('<id>') (SQL): eventos do pipeline (execuções, fluxos, qualidade de dados).
# - details:flow_progress.data_quality.expectations (SQL): extrai um campo de dentro de um texto JSON.
# - from_json(texto, schema) (SQL): converte o JSON numa estrutura; aqui, array de structs.
# - LATERAL VIEW explode(lista) (SQL): transforma cada item da lista numa linha.
# - DataFrame.groupBy(...).agg(...): agrupa e agrega (sum soma, max pega o maior valor).
# - F.regexp_replace(coluna, regex, novo): troca texto; aqui tira o sufixo "_validos" do nome.
# - F.lit(valor) / F.current_timestamp(): valor constante / momento atual.
# - DataFrame.createOrReplaceTempView(nome): expõe o DataFrame ao SQL com esse nome.
# - MERGE INTO ... USING ... ON ... WHEN MATCHED / NOT MATCHED (SQL): upsert idempotente.
# - spark.table(nome).count(): número de linhas da tabela.
from pyspark.sql import functions as F

eventos = spark.sql(f"""
SELECT origin.update_id AS update_id, timestamp AS ts,
       exp.dataset AS dataset, exp.name AS regra,
       exp.passed_records AS aprovados, exp.failed_records AS reprovados
FROM (
  SELECT origin, timestamp,
         from_json(details:flow_progress.data_quality.expectations,
                   'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>') AS exps
  FROM event_log('{PIPELINE_ID}')
  WHERE event_type = 'flow_progress'
    AND details:flow_progress.data_quality IS NOT NULL
)
LATERAL VIEW explode(exps) t AS exp
""")

novo = (
    eventos.groupBy("update_id", "dataset", "regra")
    .agg(
        F.max("ts").alias("atualizado_em"),
        F.sum("aprovados").alias("registros_aprovados"),
        F.sum("reprovados").alias("registros_reprovados"),
    )
    .withColumns(
        {
            "pipeline_id": F.lit(PIPELINE_ID),
            "entidade": F.regexp_replace("dataset", "_validos$", ""),
            "carregado_em": F.current_timestamp(),
        }
    )
)
novo.createOrReplaceTempView("dq_novo")

spark.sql(f"""
MERGE INTO {TABELA_DQ} AS destino
USING dq_novo AS origem
ON destino.update_id = origem.update_id
   AND destino.dataset = origem.dataset
   AND destino.regra = origem.regra
WHEN MATCHED THEN UPDATE SET
  destino.atualizado_em = origem.atualizado_em,
  destino.registros_aprovados = origem.registros_aprovados,
  destino.registros_reprovados = origem.registros_reprovados
WHEN NOT MATCHED THEN INSERT (
  pipeline_id, update_id, atualizado_em, dataset, entidade, regra,
  registros_aprovados, registros_reprovados, carregado_em
) VALUES (
  origem.pipeline_id, origem.update_id, origem.atualizado_em, origem.dataset, origem.entidade, origem.regra,
  origem.registros_aprovados, origem.registros_reprovados, origem.carregado_em
)
""")

print(f"regras lidas do log : {novo.count()}")
print(f"linhas na tabela    : {spark.table(TABELA_DQ).count()}")

## Resumo da última execução

Mostra, por entidade, só as regras que tiveram algum registro reprovado na execução mais recente que mediu qualidade. Execução sem dados novos não mede nada, então a "última" é a mais recente com métricas.

In [0]:
# O QUE FAZ: exibe as regras com registros reprovados na execução mais recente que tem métricas.
#
# FUNÇÕES NATIVAS:
# - spark.table(nome): abre uma tabela como DataFrame.
# - DataFrame.agg(F.max(coluna)).first()[0]: maior valor da coluna, como valor Python.
# - DataFrame.where(condição): filtra linhas.
# - DataFrame.orderBy(...): ordena.
# - display(df): mostra o DataFrame como tabela interativa do Databricks.
dq = spark.table(TABELA_DQ)
ultimo_update = (
    dq.groupBy("update_id").agg(F.max("atualizado_em").alias("m")).orderBy(F.col("m").desc()).first()["update_id"]
)
display(
    dq.where((F.col("update_id") == ultimo_update) & (F.col("registros_reprovados") > 0))
    .select("entidade", "regra", "registros_aprovados", "registros_reprovados")
    .orderBy("entidade", "regra")
)